In [ ]:
!uv pip install -q jiwer

In [ ]:
!uv pip install -q openai-whisper openai faster-whisper mistral-common pydub transformers==4.57.6
import whisper
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline, VoxtralForConditionalGeneration, AutoModelForCausalLM

In [ ]:
import jiwer
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
df = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Videos Entrevistas Medicas Spanish/Videos_Links_Transcriptions.parquet')
df

**Transcription with Fine-Tuned Model**

---



In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "/content/drive/MyDrive/Colab Notebooks/Videos Entrevistas Medicas Spanish/FT_Whisper_8/whisper-large-v3-finetuned/final_model"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    dtype=torch_dtype,
    low_cpu_mem_usage = True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model = model,
    tokenizer = processor.tokenizer,
    feature_extractor = processor.feature_extractor,
    # chunk_length_s = 30,   # Using `chunk_length_s` is very experimental with seq2seq models
    return_timestamps = True,
    batch_size = 16,  # batch size for inference - set based on your device
    dtype = torch_dtype,
    device = device,
)


df['transcription_whisper_FTed'] = ''
for index, row in df.iterrows():
  print(f"Now processing: {row['code']}")

  audio_file = f"/content/drive/MyDrive/Colab Notebooks/Videos Entrevistas Medicas Spanish/01_Audios/{row['code']}.mp3"

  result = pipe(audio_file,
                generate_kwargs = {"language": "spanish",
                                   "temperature": 0,
                                   "num_beams": 5,
                                   "use_cache": True,
                                   "do_sample": False})

  df.loc[index, 'transcription_whisper_FTed'] = result['text']

**Test one video/transcription only - to check all work**

---



In [ ]:
# ----------------------------
# 1. Evaluación de transcripción (WER)
# ----------------------------
reference_text = df['transcription_human'].iloc[0]
hypothesis_text = df['transcription_whisper_FTed'].iloc[0]

# Definir pipeline de normalización
transform = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.Strip(),
    jiwer.RemoveMultipleSpaces()
])

# Aplicar las transformaciones
reference_processed = transform(reference_text)
hypothesis_processed = transform(hypothesis_text)

# Calcular WER
wer = jiwer.wer(reference_processed, hypothesis_processed)
print(f"Word Error Rate (WER): {wer:.2%}")

print(f"Reference: {reference_processed}")
print(f"Hypothesis: {hypothesis_processed}")

In [ ]:
# -----------------------------
# 1. Define the models to compare
# -----------------------------
models = [
    'transcription_whisper_large',
    'transcription_whisper_large_v3_turbo',
    'transcription_whisper_large_v3',
    'Voxtral_Mini_3B_2507',
    'canary_1b_v2',
    'transcription_whisper_FTed',
]

# -----------------------------
# 2. Define text normalization
# -----------------------------
transform = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.Strip(),
    jiwer.RemoveMultipleSpaces()
])

# -----------------------------
# 3. Function to compute WER
# -----------------------------
def compute_wer(ref, hyp):
    if not isinstance(ref, str) or not isinstance(hyp, str):
        return None
    ref_t = transform(ref)
    hyp_t = transform(hyp)
    try:
        return jiwer.wer(ref_t, hyp_t)
    except Exception:
        return None

# -----------------------------
# 4. Compute WER for each model
# -----------------------------
wer_results = {}

for model in models:
    wer_col = f"WER_{model}"
    df[wer_col] = df.apply(lambda row: compute_wer(row['transcription_human'], row[model]), axis=1)
    wer_results[model] = df[wer_col].mean()

# -----------------------------
# 5. Create summary dataframe
# -----------------------------
wer_summary = (
    pd.DataFrame.from_dict(wer_results, orient='index', columns=['Mean_WER'])
    .sort_values('Mean_WER')
)

# -----------------------------
# 6. Display summary and details
# -----------------------------
print("\n=== Mean WER per model ===")
print(wer_summary)

print("\n=== Individual WER results (first 10 rows) ===")
cols_to_show = ['code', 'title', 'country', 'length_min'] + [f"WER_{m}" for m in models]
df[cols_to_show].head(10)

In [ ]:
# -----------------------------
# 1. Function to compute CER
# -----------------------------
def compute_cer(ref, hyp):
    if not isinstance(ref, str) or not isinstance(hyp, str):
        return None
    try:
        return jiwer.cer(transform(ref), transform(hyp))
    except Exception:
        return None

# -----------------------------
# 2. Compute CER for each model
# -----------------------------
cer_results = {}

for model in models:
    col_name = f"CER_{model}"
    df[col_name] = df.apply(lambda row: compute_cer(row['transcription_human'], row[model]), axis=1)
    cer_results[model] = df[col_name].mean()

# -----------------------------
# 3. Summary
# -----------------------------
cer_summary = (
    pd.DataFrame.from_dict(cer_results, orient='index', columns=['Mean_CER'])
    .sort_values('Mean_CER')
)

print("\n=== Mean CER per model ===")
print(cer_summary)


In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download('punkt', quiet=True)

# -----------------------------
# 1. Function to compute BLEU
# -----------------------------
def compute_bleu(ref, hyp):
    if not isinstance(ref, str) or not isinstance(hyp, str):
        return None
    ref_tokens = transform(ref).split()
    hyp_tokens = transform(hyp).split()
    try:
        return sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=SmoothingFunction().method1)
    except Exception:
        return None

# -----------------------------
# 2. Compute BLEU for each model
# -----------------------------
bleu_results = {}

for model in models:
    col_name = f"BLEU_{model}"
    df[col_name] = df.apply(lambda row: compute_bleu(row['transcription_human'], row[model]), axis=1)
    bleu_results[model] = df[col_name].mean()

# -----------------------------
# 3. Summary
# -----------------------------
bleu_summary = (
    pd.DataFrame.from_dict(bleu_results, orient='index', columns=['Mean_BLEU'])
    .sort_values('Mean_BLEU', ascending=False)
)

print("\n=== Mean BLEU per model ===")
print(bleu_summary)

In [ ]:
!uv pip install -q rouge_score

In [ ]:
from rouge_score import rouge_scorer

# -----------------------------
# 1. Function to compute ROUGE-L
# -----------------------------
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def compute_rougeL(ref, hyp):
    if not isinstance(ref, str) or not isinstance(hyp, str):
        return None
    try:
        scores = scorer.score(transform(ref), transform(hyp))
        return scores['rougeL'].fmeasure
    except Exception:
        return None

# -----------------------------
# 2. Compute ROUGE-L for each model
# -----------------------------
rouge_results = {}

for model in models:
    col_name = f"ROUGE_L_{model}"
    df[col_name] = df.apply(lambda row: compute_rougeL(row['transcription_human'], row[model]), axis=1)
    rouge_results[model] = df[col_name].mean()

# -----------------------------
# 3. Summary
# -----------------------------
rouge_summary = (
    pd.DataFrame.from_dict(rouge_results, orient='index', columns=['Mean_ROUGE_L'])
    .sort_values('Mean_ROUGE_L', ascending=False)
)

print("\n=== Mean ROUGE-L per model ===")
print(rouge_summary)

In [ ]:
!uv pip install -q evaluate bert_score

In [ ]:
import evaluate
bertscore = evaluate.load("bertscore")

# -----------------------------
# 1. Function to compute BERTScore (F1)
# -----------------------------
def compute_bertscore(ref, hyp):
    if not isinstance(ref, str) or not isinstance(hyp, str):
        return None
    try:
        result = bertscore.compute(predictions=[hyp], references=[ref], lang="es")
        return result["f1"][0]
    except Exception:
        return None

# -----------------------------
# 2. Compute BERTScore for each model
# -----------------------------
bertscore_results = {}

for model in models:
    col_name = f"BERTScore_{model}"
    df[col_name] = df.apply(lambda row: compute_bertscore(row['transcription_human'], row[model]), axis=1)
    bertscore_results[model] = df[col_name].mean()

# -----------------------------
# 3. Summary
# -----------------------------
bertscore_summary = (
    pd.DataFrame.from_dict(bertscore_results, orient='index', columns=['Mean_BERTScore'])
    .sort_values('Mean_BERTScore', ascending=False)
)

print("\n=== Mean BERTScore per model ===")
print(bertscore_summary)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# -----------------------------
# 1. Load multilingual model
# -----------------------------
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# -----------------------------
# 2. Function to compute cosine similarity
# -----------------------------
def compute_semantic_similarity(ref, hyp):
    if not isinstance(ref, str) or not isinstance(hyp, str):
        return None
    try:
        emb = embedder.encode([transform(ref), transform(hyp)], convert_to_tensor=False)
        return float(cosine_similarity([emb[0]], [emb[1]])[0][0])
    except Exception:
        return None

# -----------------------------
# 3. Compute similarity for each model
# -----------------------------
semantic_results = {}

for model in models:
    col_name = f"SemanticSim_{model}"
    df[col_name] = df.apply(lambda row: compute_semantic_similarity(row['transcription_human'], row[model]), axis=1)
    semantic_results[model] = df[col_name].mean()

# -----------------------------
# 4. Summary
# -----------------------------
semantic_summary = (
    pd.DataFrame.from_dict(semantic_results, orient='index', columns=['Mean_Semantic_Similarity'])
    .sort_values('Mean_Semantic_Similarity', ascending=False)
)

print("\n=== Mean Semantic Similarity per model ===")
print(semantic_summary)

In [ ]:
df.to_excel('/content/drive/MyDrive/Colab Notebooks/Videos Entrevistas Medicas Spanish/FT_Whisper_8/Videos_Links_Transcriptions_Metrics.xlsx')
df.to_parquet('/content/drive/MyDrive/Colab Notebooks/Videos Entrevistas Medicas Spanish/FT_Whisper_8/Videos_Links_Transcriptions_Metrics.parquet')
df.to_pickle('/content/drive/MyDrive/Colab Notebooks/Videos Entrevistas Medicas Spanish/FT_Whisper_8/Videos_Links_Transcriptions_Metrics.pkl')